In [ ]:
# SISO 5G gNB-UE Simulation using AIRSTRAN D 2200
import sys
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"

# Add src directory to Python path
sys.path.append(os.path.abspath('../src'))

# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import mitsuba as mi
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", message="invalid value encountered in multiply")
warnings.filterwarnings("ignore", category=UserWarning, module="jupyter_client")

# Import relevant components from Sionna RT
from sionna.rt import load_scene, Transmitter, Receiver, Camera, PathSolver
from sionna.rt import AntennaArray, PlanarArray, SceneObject, ITURadioMaterial
from sionna.rt.antenna_pattern import antenna_pattern_registry

scene_xml_path = "../scene/scenes/Duke/scene.xml"
scene = load_scene(scene_xml_path)

In [ ]:
# ============================================
# SISO Configuration: gNB to UE
# ============================================

scene.frequency = 3.7e9  # 3.7 GHz

# Define UE position (fixed to start)
ue_position = [10.0, 0.0, 0.0]   # UE position (x, y, z in meters)

# ============================================
# Antenna Configuration
# ============================================

# gNB antenna: 3GPP TR 38.901 pattern (AIRSTRAN D 2200)
gnb_pattern_factory = antenna_pattern_registry.get("tr38901")
gnb_pattern = gnb_pattern_factory(polarization="V")

# Friendly jammers use 3GPP TR 38.901 directional pattern so their boresight
# orientation is jointly optimised alongside position and power.
friendly_jammer = antenna_pattern_registry.get("iso")
friendly_pattern = friendly_jammer(polarization="V")

# Rx pattern used to take measurements
ue_pattern_factory = antenna_pattern_registry.get("iso")
ue_pattern = ue_pattern_factory(polarization="V")

# SISO: Single antenna element at origin [0, 0, 0] for both TX and RX
single_element = np.array([[0.0, 0.0, 0.0]])  # Shape: (1, 3)

# Configure antenna arrays
scene.tx_array = AntennaArray(
    antenna_pattern=gnb_pattern,
    normalized_positions=single_element.T  # Shape: (3, 1)
)

jammer_array = AntennaArray(
    antenna_pattern=friendly_pattern,
    normalized_positions=single_element.T
)

scene.rx_array = AntennaArray(
    antenna_pattern=ue_pattern,
    normalized_positions=single_element.T  # Shape: (3, 1)
)

# ============================================
# Add Receiver to Scene
# ============================================

# Create UE receiver
rx = Receiver(name="ue", position=ue_position, display_radius=0.03)
scene.add(rx)

# ============================================
# Configure Propagation Environment
# ============================================

# Disable scattering for basic simulation
for radio_material in scene.radio_materials.values():
    radio_material.scattering_coefficient = 0.4


In [ ]:
# Transmitters are created by setup_bs_transmitters in the zone/seeding cell.
# Instantiate path solver here for any visualisation you want to run later.
p_solver = PathSolver()

# Camera — repositioned after BS seeding
cam = Camera(position=(100.0, 100.0, 50.0))
cam.look_at([0.0, 0.0, 0.0])

# Uncomment to preview after BSs are placed:
# paths = p_solver(scene=scene, max_depth=5, los=True,
#                  specular_reflection=True, diffuse_reflection=True,
#                  refraction=False, seed=41)
# scene.preview(paths=paths, resolution=[1000, 1000], clip_at=200)

In [ ]:
from boresight_pathsolver import create_zone_mask
from multi_tx_optimizer import setup_bs_transmitters, seed_jammer_positions
import numpy as np

map_config = {
    'center': [0.0, 0.0, 0.0],
    'size': [1400, 1400],
    'cell_size': (0.5, 0.5),
    'ground_height': 0.0,
}

extent = [
    map_config['center'][0] - map_config['size'][0] / 2,
    map_config['center'][0] + map_config['size'][0] / 2,
    map_config['center'][1] - map_config['size'][1] / 2,
    map_config['center'][1] + map_config['size'][1] / 2,
]

# Organic blob zone — Fourier superposition of low-frequency harmonics over a
# base circle, like a shape drawn loosely with one finger.
rng = np.random.default_rng(seed=83)
n_pts = 120
theta = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
base_r = 180.0
r = np.full(n_pts, base_r)
for freq, (amp_lo, amp_hi) in [(2, (50, 80)), (3, (35, 60)), (4, (20, 38)), (5, (10, 22))]:
    amp = rng.uniform(amp_lo, amp_hi)
    phase = rng.uniform(0, 2 * np.pi)
    r += amp * np.cos(freq * theta + phase)
r = np.clip(r, 65, 295)

cx, cy = 0.0, -200.0
splat_vertices = [
    (round(cx + r[i] * np.cos(theta[i]), 2), round(cy + r[i] * np.sin(theta[i]), 2))
    for i in range(n_pts)
]

zone_params = {
    'center': [cx, cy],
    'vertices': splat_vertices,
}

# ── Seed BSs inside zone and register them in the scene ───────────────────────
n_bs = 2
bs_height = 50.0

print(f"Seeding {n_bs} BSs with LOS filter and farthest-point spread:")
tx_configs, bs_positions_xyz = setup_bs_transmitters(
    scene=scene,
    zone_params=zone_params,
    n_bs=n_bs,
    scene_xml_path=scene_xml_path,
    project_to_edge=True,
    bs_height=bs_height,
    target_z=1.5,
    name_prefix="bs",
    seed=44,
)
gnb_position = bs_positions_xyz[0]

# ── Seed jammer positions outside zone ────────────────────────────────────────
# n_jammers=None → one per boundary feature: concave corners + convex arc peaks.
# max_gap_deg=90 → fill jammers are added wherever the perimeter gap exceeds 90°,
# ensuring no large arc of the zone boundary is left unguarded.
jam_positions = seed_jammer_positions(
    zone_params,
    n_jammers=None,
    bs_positions=[[p[0], p[1]] for p in bs_positions_xyz],
    standoff_distance=150.0,
    min_bs_distance=75.0,
    concave_order=8,
    max_gap_deg=90.0,
    interpolation_factor=2,
    seed=22,
)
print(f"\nSeeded {len(jam_positions)} jammers outside zone (concave corners + convex arc peaks + gap fill):")
for k, (jx, jy) in enumerate(jam_positions):
    print(f"  jam_{k+1}: ({jx:.1f}, {jy:.1f})")

zone_mask, naive_look_at, zone_stats = create_zone_mask(
    map_config=map_config,
    zone_type='polygon',
    origin_point=gnb_position,
    zone_params=zone_params,
    target_height=1.5,
    scene_xml_path=scene_xml_path,
    exclude_buildings=True,
)
print(f"\nZone contains {zone_stats['num_cells']} grid cells")
print(f"Zone coverage: {zone_stats['coverage_fraction']*100:.1f}% of map")
print(f"Naive baseline look-at: {zone_stats['look_at_xyz']}")
print(f"Zone centroid: {zone_stats['centroid_xy']}")

In [ ]:
# tx_configs is returned by setup_bs_transmitters in the zone/seeding cell above.
# Names are auto-generated as "bs_0", "bs_1", ... matching n_bs.
# This cell is kept as a reference for the manual TxConfig workflow.

print(f"Using {len(tx_configs)} TxConfig(s):")
for cfg in tx_configs:
    print(f"  {cfg.name}  zone_params={'polygon' if 'vertices' in cfg.zone_params else 'box'}")

In [ ]:
from boresight_pathsolver import visualize_multi_tx_strata
import matplotlib.pyplot as plt

def strata_callback(iteration, tx_states, tx_configs, jam_positions=None):
    fig = visualize_multi_tx_strata(
        tx_states, tx_configs, map_config, iteration=iteration,
        jam_positions=jam_positions,
    )
    plt.show()
    plt.close(fig)


In [ ]:
from multi_tx_optimizer import optimize_multi_tx, JammerConfig
from angle_utils import azimuth_elevation_to_yaw_pitch
import mitsuba as mi

# One JammerConfig per seeded position (concave corners + convex arc peaks + gap fill).
jam_configs = [
    JammerConfig(name=f"jam_{k+1}", initial_power_dbm=20.0, initial_position=pos)
    for k, pos in enumerate(jam_positions)
]
print(f"Created {len(jam_configs)} jammers ({len(jam_positions)} seeded positions)")

# ── Hyperparameters ───────────────────────────────────────────────────────────
#
#   lambda_in     : penalises interior SINR holes below (gamma_db - 10 dB) when jammers present
#   lambda_out    : penalises leakage outside the zone above gamma_db (squared hinge)
#   lambda_min_j  : gate sparsity pressure — drives unused jammers off
#   lambda_spread : penalises pairwise jammer proximity; pushes jammers to distinct
#                   angular sectors around the perimeter rather than clustering
#   spread_min_dist : reference separation (m) for L_spread — penalty is 0.5 at
#                   this distance and approaches 1.0 as jammers collide
#   gamma_db      : threshold; outside cells above this contribute hinge gradient
#
_base_hparams = dict(
    # sampling
    num_sample_points=300,
    lds='Halton',
    sampler='rejection',
    sampling_strata='full',
    # signal containment
    gamma_db=0.0,
    min_sinr_db=12.0,
    lambda_in=1.0,
    lambda_out=5.0,
    # regularisation
    lambda_uniform=0.3,
    lambda_min_j=3.0,
    lambda_spread=2.0,      # spread penalty weight; 0.0 disables it
    spread_min_dist=0.0,  # 0.5 penalty when jammers are this many metres apart
    # optimiser
    learning_rate=0.6,
    num_iterations=200,
    noise_power=1e-10,
    outside_half_size=350.0
)

_shared_kwargs = dict(
    tx_configs=tx_configs,
    map_config=map_config,
    scene_xml_path=scene_xml_path,
    on_iteration_callback=strata_callback,
    **_base_hparams,
)

# ── Run 1: In-zone signal optimization (no leakage penalty) ──────────────────
print("\n" + "=" * 60)
print("RUN 1: In-zone signal optimization")
print("=" * 60)
result_bs, _ = optimize_multi_tx(
    scene=scene,
    **{**_shared_kwargs, "lambda_out": 4.0, "num_iterations": 1},
)

# ── Warm-start Run 2 from Run 1 results ───────────────────────────────────────
for cfg in tx_configs:
    r = result_bs[cfg.name]
    pos = r["final_position"]
    scene.get(cfg.name).position = mi.Point3f(float(pos[0]), float(pos[1]), float(pos[2]))
    cfg.initial_azimuth_deg, cfg.initial_elevation_deg = [float(a) for a in r["best_angles"]]

# ── Run 2: Joint BS + jammer optimization ────────────────────────────────────
print("\n" + "=" * 60)
print("RUN 2: Joint BS + jammer optimization (full DOF)")
print("=" * 60)
result_jam, jam_scene = optimize_multi_tx(
    scene=scene,
    jammer_array=jammer_array,
    jam_configs=jam_configs,
    **_shared_kwargs,
)

In [ ]:
from multi_tx_optimizer import compare_multi_tx_performance
import matplotlib.pyplot as plt

zone_masks_dict = {cfg.name: zone_mask for cfg in tx_configs}
_cmp_kwargs = dict(
    scene=scene, tx_configs=tx_configs, map_config=map_config,
    zone_masks=zone_masks_dict, noise_power=1e-10, gamma_db=0.0,
)

# ── Evaluate: BS + Jammers ────────────────────────────────────────────────────
print("=" * 60)
print("EVALUATION: BS + Jammer result")
print("=" * 60)
_, _, stats_jam = compare_multi_tx_performance(
    multi_result=result_jam, jam_scene=jam_scene, jammer_configs=jam_configs,
    **_cmp_kwargs
)
plt.show()

# ── Containment summary ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("CONTAINMENT SUMMARY: BS + Jammers")
print(f"{'TX':<8} {'metric':<14} {'BS+Jam':>10}")
print("-" * 35)
for cfg in tx_configs:
    jam = stats_jam[cfg.name].get('containment_jam', stats_jam[cfg.name]['containment'])
    print(f"{cfg.name:<8} {'rho_leak':<14} {100*jam['rho_leak']:>9.1f}%")
    print(f"{cfg.name:<8} {'rho_hole':<14} {100*jam['rho_hole']:>9.1f}%")